In [1]:
import os
import requests
from dotenv import load_dotenv

# Load API key from the project .env file
load_dotenv("../.env")
load_dotenv(".env")

api_key = os.getenv("TOMTOM_API_KEY")

if not api_key:
    print("API key nahi mili. Check the .env file location.")
else:
    url = "https://api.tomtom.com/traffic/services/4/flowSegmentData/absolute/10/json"

    params = {
        "point": "18.5204,73.8567",
        "unit": "KMPH",
        "key": api_key
    }

    response = requests.get(url, params=params, timeout=20)
    print("HTTP status:", response.status_code)

    if response.ok:
        flow = response.json()["flowSegmentData"]
        print("Current speed:", flow.get("currentSpeed"), "km/h")
        print("Free-flow speed:", flow.get("freeFlowSpeed"), "km/h")
        print("Traffic data received successfully!")
    else:
        print("API request failed:", response.status_code)
        print(response.text[:300])

HTTP status: 200
Current speed: 12 km/h
Free-flow speed: 14 km/h
Traffic data received successfully!


In [2]:
# Calculate a congestion-based traffic score
current_speed = 12
free_flow_speed = 14

if free_flow_speed > 0:
    traffic_score = min(
        100,
        max(0, (current_speed / free_flow_speed) * 100)
    )
else:
    traffic_score = None

print("Traffic Score:", round(traffic_score, 2))

Traffic Score: 85.71


In [3]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv
from pathlib import Path

# Load the API key
load_dotenv("../.env")
load_dotenv(".env")
api_key = os.getenv("TOMTOM_API_KEY")

if not api_key:
    raise ValueError("TomTom API key not found in .env")

# Fetch live traffic for the Pune test point
url = "https://api.tomtom.com/traffic/services/4/flowSegmentData/absolute/10/json"
params = {
    "point": "18.5204,73.8567",
    "unit": "KMPH",
    "key": api_key
}

response = requests.get(url, params=params, timeout=20)
response.raise_for_status()
flow = response.json()["flowSegmentData"]

current_speed = flow.get("currentSpeed")
free_flow_speed = flow.get("freeFlowSpeed")

traffic_score = (
    min(100, max(0, current_speed / free_flow_speed * 100))
    if free_flow_speed and current_speed is not None
    else None
)

# Build one real-data record
df = pd.DataFrame([{
    "latitude": 18.5204,
    "longitude": 73.8567,
    "current_speed_kmph": current_speed,
    "free_flow_speed_kmph": free_flow_speed,
    "traffic_score": round(traffic_score, 2) if traffic_score is not None else None,
    "data_source": "TomTom Traffic API"
}])

# Save output
out_dir = Path("../data/processed")
out_dir.mkdir(parents=True, exist_ok=True)
df.to_csv(out_dir / "traffic_test_data.csv", index=False)

display(df)
print("Saved to:", (out_dir / "traffic_test_data.csv").resolve())

,latitude,longitude,current_speed_kmph,free_flow_speed_kmph,traffic_score,data_source
0,18.5204,73.8567,12,15,80.0,TomTom Traffic API


Saved to: C:\Users\Admin\OneDrive\Desktop\TY Project\data\processed\traffic_test_data.csv


In [4]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv
from pathlib import Path
from time import sleep

load_dotenv("../.env")
load_dotenv(".env")
api_key = os.getenv("TOMTOM_API_KEY")

# Five sample road points around Pune
points = [
    (18.5204, 73.8567),
    (18.5314, 73.8446),
    (18.5074, 73.8077),
    (18.5679, 73.9143),
    (18.4575, 73.8677)
]

records = []

for lat, lon in points:
    url = "https://api.tomtom.com/traffic/services/4/flowSegmentData/absolute/10/json"
    params = {
        "point": f"{lat},{lon}",
        "unit": "KMPH",
        "key": api_key
    }

    try:
        r = requests.get(url, params=params, timeout=20)
        r.raise_for_status()
        flow = r.json()["flowSegmentData"]

        current = flow.get("currentSpeed")
        free = flow.get("freeFlowSpeed")
        score = min(100, max(0, current / free * 100)) if free and current is not None else None

        records.append({
            "latitude": lat,
            "longitude": lon,
            "current_speed_kmph": current,
            "free_flow_speed_kmph": free,
            "traffic_score": round(score, 2) if score is not None else None,
            "data_source": "TomTom Traffic API"
        })

    except requests.RequestException as e:
        print(f"Failed for {lat}, {lon}: {e}")

    sleep(0.2)

df_traffic = pd.DataFrame(records)
out = Path("../data/processed")
out.mkdir(parents=True, exist_ok=True)
df_traffic.to_csv(out / "traffic_pune_sample.csv", index=False)

display(df_traffic)
print("Successful records:", len(df_traffic))
print("Saved:", (out / "traffic_pune_sample.csv").resolve())

,latitude,longitude,current_speed_kmph,free_flow_speed_kmph,traffic_score,data_source
0,18.5204,73.8567,12,15,80.00,TomTom Traffic API
1,18.5314,73.8446,24,24,100.00,TomTom Traffic API
2,18.5074,73.8077,27,36,75.00,TomTom Traffic API
3,18.5679,73.9143,25,35,71.43,TomTom Traffic API
4,18.4575,73.8677,18,18,100.00,TomTom Traffic API


Successful records: 5
Saved: C:\Users\Admin\OneDrive\Desktop\TY Project\data\processed\traffic_pune_sample.csv


In [5]:
# Add congestion percentage and collection timestamp
from datetime import datetime, timezone

df_traffic["congestion_percent"] = (
    100 * (
        1 - df_traffic["current_speed_kmph"]
        / df_traffic["free_flow_speed_kmph"]
    )
).clip(0, 100).round(2)

df_traffic["collected_at_utc"] = datetime.now(
    timezone.utc
).isoformat()

df_traffic.to_csv(
    "../data/processed/traffic_pune_sample.csv",
    index=False
)

display(df_traffic)
print("Updated traffic sample saved.")

,latitude,longitude,current_speed_kmph,free_flow_speed_kmph,traffic_score,data_source,congestion_percent,collected_at_utc
0,18.5204,73.8567,12,15,80.00,TomTom Traffic API,20.00,2026-10-23T07:20:25.429211+00:00
1,18.5314,73.8446,24,24,100.00,TomTom Traffic API,0.00,2026-10-23T07:20:25.429211+00:00
2,18.5074,73.8077,27,36,75.00,TomTom Traffic API,25.00,2026-10-23T07:20:25.429211+00:00
3,18.5679,73.9143,25,35,71.43,TomTom Traffic API,28.57,2026-10-23T07:20:25.429211+00:00
4,18.4575,73.8677,18,18,100.00,TomTom Traffic API,0.00,2026-10-23T07:20:25.429211+00:00


Updated traffic sample saved.


In [1]:
import os
import shutil

os.makedirs("../data/final", exist_ok=True)

shutil.copy2(
    "../data/processed/traffic_pune_sample.csv",
    "../data/final/traffic_pune_sample.csv"
)

print("Traffic dataset saved in data/final!")

Traffic dataset saved in data/final!
